In [0]:
dbutils.widgets.text("catalog","dev_l2")
catalog = dbutils.widgets.get("catalog")

In [0]:
spark.sql(f"use catalog {catalog}")

In [0]:
from pyspark.sql import functions as F

In [0]:
def valid_pk(col):
    return F.col(col).isNotNull()

def valid_date(col):
    return (F.col(col) >= F.to_date(F.lit('1992-01-01'))) & (F.col(col) <= F.to_date(F.lit('1998-12-31')))

def valid_fk(col,ref_col, ref):
    return (F.col(col).isNotNull()) & F.col(col).isin([row[ref_col] for row in ref.select(ref_col).distinct().collect()])

In [0]:
region = spark.read.table("bronze.region")

In [0]:
region_std = region.select(
    "r_regionkey",
    F.trim(F.upper(F.col("r_name"))).alias("r_name"),
    F.trim(F.col("r_comment")).alias("r_comment")
)

In [0]:
valid_region = region_std.filter(valid_pk("r_regionkey"))
valid_region.write.mode("overwrite").saveAsTable("silver.region")

In [0]:
reject_reasons = F.array_compact(
    F.array(F.when(~valid_pk("r_regionkey"), "INVLAID_PRIMARY_KEY"))
)

quarantine_region = region_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0)
quarantine_region.write.mode("overwrite").saveAsTable("silver.quarantine_region")

In [0]:
nation = spark.read.table("bronze.nation")
region = spark.read.table("silver.region")


In [0]:
nation_std = nation.select(
    "n_nationkey",
    F.trim(F.upper(F.col("n_name"))).alias("n_name"),
    F.col("n_regionkey"),
    F.trim(F.col("n_comment")).alias("n_comment")
).dropDuplicates()

In [0]:
valid_nations = nation_std.filter(valid_pk("n_nationkey") & valid_fk("n_regionkey", "r_regionkey", region))
valid_nations.write.mode("overwrite").saveAsTable("silver.nation")

In [0]:
reject_reasons = F.array_compact(
    F.array(
        F.when(~valid_pk("n_nationkey"), "INVLAID_PRIMARY_KEY"),
        F.when(~valid_fk("n_regionkey", "r_regionkey", region), "INVALID_REFERENCE_KEY")
    )
)

quarantine_nation = nation_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0)
quarantine_nation.write.mode("overwrite").saveAsTable("silver.quarantine_nation")

In [0]:
customer = spark.read.table("bronze.customer")
nation = spark.read.table("silver.nation")


In [0]:
customer_std = customer.select(
    "c_custkey",
    F.trim(F.initcap(F.col("c_name"))).alias("c_name"),
    F.trim(F.col("c_address")).alias("c_address"),
    "c_nationkey",
    F.trim(F.col("c_phone")).alias("c_phone"),
    "c_acctbal",
    F.trim(F.upper(F.col("c_mktsegment"))).alias("c_mktsegment"),
    F.trim(F.col("c_comment")).alias("c_comment")
) .dropDuplicates()

In [0]:
valid_customer = customer_std.filter(valid_pk("c_custkey") & valid_fk("c_nationkey", "n_nationkey", nation))
valid_customer.write.mode("overwrite").saveAsTable("silver.customer")

In [0]:
reject_reasons = F.array_compact(
    F.array(
        F.when(~valid_pk("c_custkey"), "INVLAID_PRIMARY_KEY"),
        F.when(~valid_fk("c_nationkey", "n_nationkey", nation), "INVALID_REFERENCE_KEY")
    )
)
quarantine_customer = customer_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0)
quarantine_customer.write.mode("overwrite").saveAsTable("silver.quarantine_customer")

In [0]:
supplier = spark.read.table("bronze.supplier")
nation = spark.read.table("silver.nation")

In [0]:
supplier_std = supplier.select (
    "s_suppkey",
    F.trim(F.initcap(F.col("s_name"))).alias("s_name"),
    F.trim(F.col("s_address")).alias("s_address"),
    "s_nationkey",
    F.trim(F.col("s_phone")).alias("s_phone"),
    "s_acctbal",
    F.trim(F.col("s_comment")).alias("s_comment")
) .dropDuplicates()

In [0]:
valid_supplier = supplier_std.filter(valid_pk("s_suppkey") & valid_fk("s_nationkey", "n_nationkey", nation))
valid_supplier.write.mode("overwrite").saveAsTable("silver.supplier")

In [0]:
reject_reasons = F.array_compact(
    F.array(
        F.when(~valid_pk("s_suppkey"), "INVLAID_PRIMARY_KEY"),
        F.when(~valid_fk("s_nationkey", "n_nationkey", nation), "INVALID_REFERENCE_KEY")
    )
)

quarantine_supplier = supplier_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0)
quarantine_supplier.write.mode("overwrite").saveAsTable("silver.quarantine_supplier")

In [0]:
order = spark.read.table("bronze.order")
customer = spark.read.table("silver.customer")

In [0]:
order_std = order.select(
    "o_orderkey",
    "o_custkey",
    F.upper(F.col("o_orderstatus")).alias("o_orderstatus"),
    "o_totalprice",
    "o_orderdate",
    F.trim(F.upper(F.col("o_orderpriority"))).alias("o_orderpriority"),
    F.trim(F.upper(F.col("o_clerk"))).alias("o_clerk"),
    "o_shippriority",
    F.trim(F.col("o_comment")).alias("o_comment")
).dropDuplicates()

In [0]:
valid_total_price = F.col("o_totalprice") >= 0

valid_order = order_std.filter(valid_pk("o_orderkey") & valid_fk("o_custkey", "c_custkey", customer) & valid_date("o_orderdate") & valid_total_price)
valid_order.write.mode("overwrite").saveAsTable("silver.order")

In [0]:
reject_reasons = F.array_compact(
    F.array(
        F.when(~valid_pk("o_orderkey"), "INVLAID_PRIMARY_KEY"),
        F.when(~valid_fk("o_custkey", "c_custkey", customer), "INVALID_REFERENCE_KEY"),
        F.when(~valid_date("o_orderdate"), "INVALID_DATE"),
        F.when(~valid_total_price, "INVALID_TOTAL_PRICE")
    )
)

quarantine_order = order_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0)
quarantine_order.write.mode("overwrite").saveAsTable("silver.quarantine_order")

In [0]:
part = spark.read.table("bronze.part")

In [0]:
part_std = part.select(
    "p_partkey",
    F.trim(F.col("p_name")).alias("p_name"),
    F.trim(F.col("p_mfgr")).alias("p_mfgr"),
    F.trim(F.col("p_brand")).alias("p_brand"),
    F.trim(F.col("p_type")).alias("p_type"),
    "p_size",
    F.trim(F.upper(F.col("p_container"))).alias("p_container"),
    "p_retailprice",
    F.trim(F.col("p_comment")).alias("p_comment")
).dropDuplicates()

In [0]:
valid_retail_price = F.col("p_retailprice") >= 0

valid_part = part_std.filter(valid_pk("p_partkey") & valid_retail_price)
valid_part.write.mode("overwrite").saveAsTable("silver.part")

In [0]:
reject_reasons = F.array_compact(
    F.array(
        F.when(~valid_pk("p_partkey"), "INVLAID_PRIMARY_KEY"),
        F.when(~valid_retail_price, "INVALID_RETAIL_PRICE")
    )
)

quarantine_part = part_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0)
quarantine_part.write.mode("overwrite").saveAsTable("silver.quarantine_part")

In [0]:
partsupp = spark.read.table("bronze.partsupp")
part = spark.read.table("silver.part")
supplier = spark.read.table("silver.supplier")

In [0]:
partsupp_std = partsupp.select(
    "ps_partkey",
    "ps_suppkey",
    "ps_availqty",
    "ps_supplycost",
    "ps_comment"
).dropDuplicates()

In [0]:
valid_quantity = F.col("ps_availqty") >= 0
valid_cost = F.col("ps_supplycost") > 0

valid_partsupp = partsupp_std.filter(valid_fk("ps_partkey", "p_partkey", part) & valid_fk("ps_suppkey", "s_suppkey", supplier) & valid_quantity & valid_cost)
valid_partsupp.write.mode("overwrite").saveAsTable("silver.partsupp")

In [0]:
reject_reasons = F.array_compact(
    F.array(
        F.when(~valid_fk("ps_partkey", "p_partkey", part) | ~valid_fk("ps_suppkey", "s_suppkey", supplier), "INVALID_PRIMARY_KEY"),
        F.when(~valid_quantity, "INVALID_QUANTITY"),
        F.when(~valid_cost, "INVALID_COST")
    )
)

quarantine_partsupp = partsupp_std.withColumn("reject_reasons", reject_reasons) \
    .filter(F.size(F.col("reject_reasons")) > 0)
quarantine_partsupp.write.mode("overwrite").saveAsTable("silver.quarantine_partsupp")